In [2]:
import heapq
from collections import defaultdict

VERTICES = ['A','B','C','D','E','F','G']

ARISTAS = [
    ('A','B',7),
    ('A','D',5),
    ('B','C',8),
    ('B','D',9),
    ('B','E',7),
    ('C','E',5),
    ('D','E',15),
]



class UnionFind:
    def __init__(self, nodos):
        self.padre = {n: n for n in nodos}
        self.rango = {n: 0 for n in nodos}

    def encontrar(self, x):
        if self.padre[x] != x:
            self.padre[x] = self.encontrar(self.padre[x])
        return self.padre[x]

    def unir(self, x, y):
        rx, ry = self.encontrar(x), self.encontrar(y)
        if rx == ry:
            return False
        if self.rango[rx] < self.rango[ry]:
            rx, ry = ry, rx
        self.padre[ry] = rx
        if self.rango[rx] == self.rango[ry]: # Fixed typo: 'self.rando[ry]' to 'self.rango[ry]'
            self.rango[rx] += 1
        return True


def kruskal(vertices, aristas):
    uf = UnionFind(vertices)
    aristas_ordenadas = sorted(aristas, key=lambda e:e[2])
    mst = []
    peso_total = 0

    print("==kruskal==")
    # print(f"{arista <10}") # Removed problematic print statement


    for u, v, w in aristas_ordenadas:
        if uf.unir(u,v):
            mst.append((u,v,w))
            peso_total += w
            print(f"Added edge: {u}-{v} (weight: {w})") # Added more informative print
        else:
            print(f"Skipped edge: {u}-{v} (creates cycle)") # Added more informative print

        if len(mst) == len(vertices) - 1:
            break # Fixed IndentationError: Added break statement

    print(f"peso del MST {peso_total}")
    return mst, peso_total




def prim(vertices, aristas, inicio = 'A'):
    grafo = defaultdict(list)
    for u, v, w in aristas:
        grafo[u].append((w,v))
        grafo[v].append((w,u))

    visitados = set()
    visitados.add(inicio)

    heap = [(w,v, inicio) for w, v in grafo[inicio]]
    heapq.heapify(heap)

    mst = []
    peso_total = 0


    print("PRIM")
    print(f"nodo inicial {inicio}")
    # print(f"{arista}{peso}{accion}") # Removed problematic print statement
    print("-"*35)

    while heap and len(mst)< len(vertices) - 1:
        w, v, u = heapq.heappop(heap)

        if v in visitados:
            print(f"Skipped edge {u}-{v} (visited)") # Added more informative print
            continue


        visitados.add(v)
        mst.append((u,v,w))
        peso_total += w
        print(f"Added edge: {u}-{v} (weight: {w})") # Added more informative print

        for peso_vecino, vecino in grafo[v]: # Renamed 'peso' to 'peso_vecino' to avoid potential conflicts
            if vecino not in visitados:
                heapq.heappush(heap, (peso_vecino, vecino, v))

    print(f"peso total del MST: {peso_total}")
    return mst, peso_total



if __name__ =="__main__":
    mst_k, pk = kruskal(VERTICES, ARISTAS)
    mst_p, pp = prim(VERTICES, ARISTAS, inicio='A')

    print("RESUMEN")

    print(f"kruskal: {[(u,v,w) for u,v,w in mst_k]} peso = {pk}")
    print(f"Prim aristas: {[(u,v,w) for u,v,w in mst_p]} peso = {pp}")
    assert pk == pp, "los pesos deberian ser iguales"
    print("ambos algoritmos producen el mismo peso total")

==kruskal==
Added edge: A-D (weight: 5)
Added edge: C-E (weight: 5)
Added edge: A-B (weight: 7)
Added edge: B-E (weight: 7)
Skipped edge: B-C (creates cycle)
Skipped edge: B-D (creates cycle)
Skipped edge: D-E (creates cycle)
peso del MST 24
PRIM
nodo inicial A
-----------------------------------
Added edge: A-D (weight: 5)
Added edge: A-B (weight: 7)
Added edge: B-E (weight: 7)
Added edge: E-C (weight: 5)
Skipped edge B-C (visited)
Skipped edge D-B (visited)
Skipped edge D-E (visited)
peso total del MST: 24
RESUMEN
kruskal: [('A', 'D', 5), ('C', 'E', 5), ('A', 'B', 7), ('B', 'E', 7)] peso = 24
Prim aristas: [('A', 'D', 5), ('A', 'B', 7), ('B', 'E', 7), ('E', 'C', 5)] peso = 24
ambos algoritmos producen el mismo peso total


In [4]:
import heapq
from collections import Counter
from dataclasses import dataclass, field
from typing import Optional


@dataclass(order=True)
class Nodo:
    freq: int
    simbolo: Optional[str] = field(compare=False, default= None)
    izq: Optional['Nodo'] = field(compare = False, default=None)
    der: Optional['Nodo'] = field(compare=False, default=None)

    def es_hoja(self):
        return self.izq is None and self.der is None



def construir_arbol(texto: str) -> Nodo:
    frecuencia = Counter(texto)

    heap = [Nodo(freq=f, simbolo=c) for c, f in frecuencia.items()] # Fixed: 'frecuencias' to 'frecuencia'
    heapq.heapify(heap)


    if len(heap) == 1:
        # Handle single character input edge case: create a dummy root
        unico = heapq.heappop(heap)
        raiz = Nodo(freq = unico.freq, izq=unico)
        heapq.heappush(heap, raiz)


    while len(heap) > 1:
        izq = heapq.heappop(heap)
        der = heapq.heappop(heap)
        padre = Nodo(freq=izq.freq + der.freq, izq=izq, der=der)
        heapq.heappush(heap, padre)

    return heap[0]



def generar_codigos(raiz: Nodo) -> dict[str, str]: # Renamed 'rais' to 'raiz' for consistency
    codigos = {} # Initialized codigos dictionary
    def dfs(nodo: Nodo, codigo : str):
        if nodo is None:
            return
        if nodo.es_hoja():
            codigos[nodo.simbolo] = codigo or '0' # Handles single character edge case for code '0'
            return
        dfs(nodo.izq, codigo + '0')
        dfs(nodo.der, codigo + '1')

    dfs(raiz, '')
    return codigos


def codificar(texto: str, codigos: dict[str, str]) -> str:
    return ''.join(codigos[c] for c in texto) # Fixed typo: '.joint' to '.join'


def decodificar(bits: str, raiz: Nodo) -> str:
    resultado = []
    nodo = raiz
    for bit in bits:
        # Handle cases where a character might not have a left or right child yet
        if bit == '0':
            if nodo.izq is None:
                raise ValueError("Invalid Huffman code: '0' at leaf node or unexpected structure")
            nodo = nodo.izq
        else:
            if nodo.der is None:
                raise ValueError("Invalid Huffman code: '1' at leaf node or unexpected structure")
            nodo = nodo.der

        if nodo.es_hoja():
            resultado.append(nodo.simbolo)
            nodo = raiz
    return ''.join(resultado)


def mostrar_tabla(codigos: dict[str, str], frecuencia: Counter, texto: str): # Renamed 'frecuencias' to 'frecuencia'
    total_chars = len(texto)
    bits_huffman = sum(frecuencia[c] * len(cod) for c, cod in codigos.items())
    bits_ascii = total_chars * 8 # Fixed: 'bits.ascii' to 'bits_ascii' and assigned correctly

    print(f"{'Simbolo': <10} {'Freq':<6} {'Cod':<20} {'ASCII Bits':<12} {'Huffman Bits'}") # Fixed: 'simbolo' undefined and header improved
    print("-"*70)
    for c, cod in sorted(codigos.items(), key= lambda x: -frecuencia[x[0]]):
        sym = repr(c)
        print(f"{sym: <10} {frecuencia[c]:<6} {cod:<20} {8:<12} {len(cod)}") # Fixed: 'frecuencias' to 'frecuencia' and added 8 explicitly for ASCII bits per char


    ahorro = (1 - bits_huffman / bits_ascii) * 100 if bits_ascii > 0 else 0 # Added check for division by zero
    print(f"\nTexto original: {total_chars} caracteres, {bits_ascii} bits (ASCII)")
    print(f"Huffman: {bits_huffman} bits")
    print(f"Compresión: {ahorro:.2f}%") # Formatted percentage



def huffman(texto:str):
    print(f'Texto: "{texto}"')

    # Calculate frequency once and pass it around
    frecuencia = Counter(texto)

    raiz = construir_arbol(texto)
    codigos = generar_codigos(raiz)
    bits = codificar(texto, codigos)
    decoded = decodificar(bits, raiz)

    mostrar_tabla(codigos, frecuencia, texto) # Passed 'frecuencia' instead of recalculating

    print(f"\nCodificado: {bits}")
    print(f"Decodificado: {decoded}") # Fixed f-string for decoded
    assert decoded == texto, "Error de decodificación: el texto original no coincide con el decodificado"
    print("Decodificación exitosa!") # Improved message

    return codigos, bits



if __name__ == "__main__":
    print("="*60)
    print("Ejemplo 1 - Texto clásico")
    print("="*60)
    huffman("abracadabra")

    print("\n" + "="*60)
    print("Ejemplo 2 - Frecuencias dispares")
    print("="*60)
    huffman("aaaaabbbcc")

    print("\n" + "="*60)
    print("Ejemplo 3 - Frase")
    print("="*60)
    huffman("prueba huffman") # Fixed typo: 'preba' to 'prueba'

    print("\n" + "="*60)
    print("Ejemplo 4 - Caracter único")
    print("="*60)
    huffman("a")

Ejemplo 1 - Texto clásico
Texto: "abracadabra"
Simbolo    Freq   Cod                  ASCII Bits   Huffman Bits
----------------------------------------------------------------------
'a'        5      0                    8            1
'r'        2      110                  8            3
'b'        2      111                  8            3
'd'        1      100                  8            3
'c'        1      101                  8            3

Texto original: 11 caracteres, 88 bits (ASCII)
Huffman: 23 bits
Compresión: 73.86%

Codificado: 01111100101010001111100
Decodificado: abracadabra
Decodificación exitosa!

Ejemplo 2 - Frecuencias dispares
Texto: "aaaaabbbcc"
Simbolo    Freq   Cod                  ASCII Bits   Huffman Bits
----------------------------------------------------------------------
'a'        5      0                    8            1
'b'        3      11                   8            2
'c'        2      10                   8            2

Texto original: 10 cara

In [6]:
import random
import math
from collections import defaultdict
from copy import deepcopy

def construir_grafo(vertices, aristas):
    g = defaultdict(list)
    for u, v in aristas:
        g[u].append(v)
        g[v].append(u)
    return g


def contraer(grafo, u, v):
  for vecino in grafo[v]:
      # Update references to v in neighbors' adjacency lists to u
      grafo[vecino] = [u if x == v else x for x in grafo[vecino]]
      # Add edges from v's neighbors to u, making sure not to add self-loops initially
      if vecino != u: # Avoid adding self-loops during initial append
          grafo[u].append(vecino)

  # Remove self-loops from the new supernode u
  grafo[u] = [x for x in grafo[u] if x != u]
  del grafo[v]


def karger_una_vez(grafo_original):
    g = deepcopy(grafo_original)

    while len(g) > 2:
        u = random.choice(list(g.keys()))
        # Ensure g[u] is not empty before choosing a neighbor
        if not g[u]: # If u has no neighbors, it can't be contracted with anything. Skip this u.
            del g[u] # Remove isolated vertex
            if len(g) <= 2: break # Check if remaining graph is too small after removal
            continue

        v = random.choice(g[u])
        contraer(g, u, v)

    # After contraction, there should be exactly two supernodes left
    if len(g) != 2:
        # This can happen if isolated vertices were removed and the graph became too small
        # or if the graph was disconnected. For Karger's, assume connected graph.
        # If it happens, it means an invalid state for min-cut.
        # For simplicity, we can return 0 or handle as an error if graph is not strictly connected.
        return float('inf') # Indicates an error or disconnected component problem

    nodos = list(g.keys())
    # The cut is the number of edges between the two remaining supernodes
    return len(g[nodos[0]])



def karger(vertices, aristas, iteraciones=None):
    n = len(vertices)
    if iteraciones is None:
        # A larger number of iterations increases the probability of finding the min cut
        iteraciones = math.ceil(n * n * math.log(n))

    # The original graph structure is passed to karger_una_vez, not a deepcopy
    # karger_una_vez already makes a deepcopy, so `grafo` below is just for initial construction
    grafo_base = construir_grafo(vertices, aristas) # Renamed to avoid confusion with deepcopy in loop
    min_corte = float('inf')

    print(f"Karger's algorithm: {iteraciones} iterations, {n} vertices, {len(aristas)} aristas")

    for i in range(iteraciones):
        corte = karger_una_vez(grafo_base) # Pass the original structure
        if corte < min_corte:
            min_corte = corte
            # Corrected f-string for printing iteration and new minimum
            print(f"Iteration {i + 1}: new minimum cut = {min_corte}")

    return min_corte



def contraer_hasta(grafo, t):
    while len(grafo) > t:
        u = random.choice(list(grafo.keys()))
        if not grafo[u]: # Handle isolated vertices
            del grafo[u]
            continue
        v = random.choice(grafo[u])
        contraer(grafo, u, v)



def karger_stein(grafo_original):
    n = len(grafo_original)

    if n <= 6: # Base case for recursion
        # For very small graphs, run Karger's simple algorithm multiple times
        # and return the minimum cut found.
        return min(karger_una_vez(grafo_original) for _ in range(10))

    # Calculate t based on the Karger-Stein algorithm's recurrence relation
    t = math.ceil(1 + n / math.sqrt(2))

    resultados = []
    for _ in range(2):
        g = deepcopy(grafo_original)
        contraer_hasta(g, t)
        resultados.append(karger_stein(g))


    return min(resultados)



def karger_stein_repetido(vertices, aristas, iteraciones = None):
    n = len(vertices)
    if iteraciones is None:
        # The number of repetitions for Karger-Stein for high probability of success
        iteraciones = math.ceil(math.log(n) ** 2) # Often given as O(log^2 n)

    grafo_base = construir_grafo(vertices, aristas) # Base graph for deepcopy in loop
    min_corte = float('inf')

    print(f"Karger-Stein algorithm: {iteraciones} iterations, {n} vertices") # Added f-prefix


    for i in range(iteraciones):
        corte = karger_stein(deepcopy(grafo_base))
        if corte < min_corte:
            min_corte = corte
            # Corrected f-string for printing iteration and new minimum
            print(f"Iteration {i + 1}: new minimum cut = {min_corte}")

    return min_corte



if __name__ == "__main__":

    v1 = [1,2,3,4,5,6]
    a1 = [(1,2),(2,3),(1,4),(2,5),(3,6),(4,5),(5,6)] # Changed to lowercase a1


    print("="*55)
    print("Grafo 1 - corte minimo esperado: 2")
    print("="*55)
    r1 = karger(v1,a1, iteraciones=200) # Used lowercase v1 and a1
    print(f"Resultado Karger: {r1}")
    r1ks = karger_stein_repetido(v1,a1) # Used lowercase v1 and a1
    print(f"Resultado Karger-Stein: {r1ks}") # Added f-prefix


    v2 = ['A','B','C','D','E','F'] # Changed to lowercase v2
    a2 = [ # Changed to lowercase a2
        ('A','B'),('A','D'),('A','E'),
        ('B','C'),('B','D'),('B','E'),
        ('C','E'),('C','F'),
        ('D','E'),('E','F'),
    ]

    print("=" * 55)
    print("Grafo 2  —  grafo denso")
    print("=" * 55)
    r2 = karger(v2, a2, iteraciones=300) # Used lowercase v2 and a2
    print(f"\nResultado Karger        : {r2}")
    r2ks = karger_stein_repetido(v2, a2) # Used lowercase v2 and a2
    print(f"Resultado Karger-Stein  : {r2ks}")

Grafo 1 - corte minimo esperado: 2
Karger's algorithm: 200 iterations, 6 vertices, 7 aristas
Iteration 1: new minimum cut = 2
Resultado Karger: 2
Karger-Stein algorithm: 4 iterations, 6 vertices
Iteration 1: new minimum cut = 2
Resultado Karger-Stein: 2
Grafo 2  —  grafo denso
Karger's algorithm: 300 iterations, 6 vertices, 10 aristas
Iteration 1: new minimum cut = 3
Iteration 8: new minimum cut = 2

Resultado Karger        : 2
Karger-Stein algorithm: 4 iterations, 6 vertices
Iteration 1: new minimum cut = 2
Resultado Karger-Stein  : 2
